In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
%cd "/content/drive/MyDrive/Data-sci capstone: NYC Apartment Feature Valuations/Apartment_Features/"

/content/drive/MyDrive/Data-sci capstone: NYC Apartment Feature Valuations/Apartment_Features


In [19]:
import pandas as pd
%cd "/content/drive/MyDrive/Data-sci capstone: NYC Apartment Feature Valuations/Apartment_Features/Must_Have/two-sigma-connect-rental-listing-inquiries/"
df = pd.read_json("train.json")


/content/drive/MyDrive/Data-sci capstone: NYC Apartment Feature Valuations/Apartment_Features/Must_Have/two-sigma-connect-rental-listing-inquiries


In [ ]:
df_test = pd.read_json("test.json")

In [ ]:
frames = [df, df_test]
df = pd.concat(frames)
df.shape[0]
# successfully brought the test and train datasets together.
# Since we aren't doing some fancy method, probably just a simple glm? Maybe a random forest?
# I think a random forest is probably pretty good for both.

198670

In [7]:
%cd "/content/drive/MyDrive/Data-sci capstone: NYC Apartment Feature Valuations/Apartment_Features/Must_Have/"
df.to_csv('apartment_rentals_2017.csv', index=False)

/content/drive/MyDrive/Data-sci capstone: NYC Apartment Feature Valuations/Apartment_Features/Must_Have


In [8]:
# creating a dataframe for the apartment features. Will join this onto the main df when I need to.
# just starting off seperate so its a bit smaller, more efficient
# to get the rows I need to get all the apartment 'features' from the "features" column. Split on ','
# then order by highest to lowest
import numpy as np
features_ndarray = df['features'].values
features = np.concatenate(features_ndarray).reshape(-1)

In [ ]:
for i in range(len(features)):
  print(features[i])
  if i >= 20:
    break;

Dining Room
Pre-War
Laundry in Building
Dishwasher
Hardwood Floors
Dogs Allowed
Cats Allowed
Doorman
Elevator
Laundry in Building
Dishwasher
Hardwood Floors
No Fee
Doorman
Elevator
Laundry in Building
Laundry in Unit
Dishwasher
Hardwood Floors
Doorman
Elevator


In [9]:
features_dict = {}
for feature in features:
  feature = feature.lower()
  if feature not in features_dict:
    features_dict.update({feature : 1})
  else:
    feature_count = features_dict.get(feature) + 1
    features_dict.update({feature : feature_count})

In [10]:
# now we have the dict, we need to sort it by count to ensure that all the feature names are standardized
# will output first 50 large to small
features_sorted = sorted(features_dict.items(), key=lambda item: item[1], reverse=True)
for i in range(50):
  print(features_sorted[i])

('elevator', 26273)
('hardwood floors', 23558)
('cats allowed', 23540)
('dogs allowed', 22035)
('doorman', 20967)
('dishwasher', 20806)
('laundry in building', 18944)
('no fee', 18079)
('fitness center', 13257)
('laundry in unit', 9435)
('pre-war', 9149)
('roof deck', 6555)
('outdoor space', 5270)
('dining room', 5150)
('high speed internet', 4299)
('balcony', 3058)
('swimming pool', 2730)
('new construction', 2608)
('terrace', 2313)
('exclusive', 2167)
('loft', 2101)
('garden/patio', 1943)
('wheelchair access', 1358)
('prewar', 1349)
('common outdoor space', 1293)
('hardwood', 1058)
('fireplace', 919)
('simplex', 908)
('lowrise', 789)
('garage', 756)
('laundry room', 724)
('reduced fee', 699)
('furnished', 690)
('multi-level', 632)
('high ceilings', 613)
('private outdoor space', 534)
('publicoutdoor', 423)
('parking space', 418)
('roof-deck', 397)
('live in super', 366)
('renovated', 337)
('pool', 323)
('on-site laundry', 316)
('laundry', 283)
('green building', 247)
('storage', 217)

In [11]:
# need to get everyting to _tolower
# and need to strip leading and ending ' -/.'
features_dict = {}
for feature in features:
  feature = feature.lower()
  if feature not in features_dict:
    features_dict.update({feature : 1})
  else:
    feature_count = features_dict.get(feature) + 1
    features_dict.update({feature : feature_count})

In [12]:
# running sort again now on lower text
features_sorted = sorted(features_dict.items(), key=lambda item: item[1], reverse=True)
for i in range(50):
  print(features_sorted[i])

('elevator', 26273)
('hardwood floors', 23558)
('cats allowed', 23540)
('dogs allowed', 22035)
('doorman', 20967)
('dishwasher', 20806)
('laundry in building', 18944)
('no fee', 18079)
('fitness center', 13257)
('laundry in unit', 9435)
('pre-war', 9149)
('roof deck', 6555)
('outdoor space', 5270)
('dining room', 5150)
('high speed internet', 4299)
('balcony', 3058)
('swimming pool', 2730)
('new construction', 2608)
('terrace', 2313)
('exclusive', 2167)
('loft', 2101)
('garden/patio', 1943)
('wheelchair access', 1358)
('prewar', 1349)
('common outdoor space', 1293)
('hardwood', 1058)
('fireplace', 919)
('simplex', 908)
('lowrise', 789)
('garage', 756)
('laundry room', 724)
('reduced fee', 699)
('furnished', 690)
('multi-level', 632)
('high ceilings', 613)
('private outdoor space', 534)
('publicoutdoor', 423)
('parking space', 418)
('roof-deck', 397)
('live in super', 366)
('renovated', 337)
('pool', 323)
('on-site laundry', 316)
('laundry', 283)
('green building', 247)
('storage', 217)

In [17]:
# get new columns for each feature we want to track in the dataset.
# initalized to null/0 for not have, and 1 for have
# so features:
# Elevators
feature_mapping = {
    'elevator': 'elevator',
    'cats allowed': 'pet policy',
    'dogs allowed': 'pet policy',
    'pets allowed': 'pet policy',
    'pets on approval': 'pet policy',
    'doorman': 'doorman',
    'concierge': 'doorman',
    'valet': 'doorman',
    'full-time doorman': 'doorman',
    'laundry in unit': 'in unit wash/dryer',
    'washer in unit': 'in unit wash/dryer',
    'dryer in unit': 'in unit wash/dryer',
    'washer/dryer': 'in building wash/dryer',
    'laundry in building': 'in building wash/dryer',
    'on-site laundry': 'in building wash/dryer',
    'laundry': 'in building wash/dryer',
    'laundry room': 'in building wash/dryer',
    'dishwasher': 'dishwasher',
    'outdoor space': 'outdoor space',
    'balcony': 'outdoor space',
    'garden/patio': 'outdoor space',
    'roof deck': 'outdoor space',
    'roof-deck': 'outdoor space',
    'private outdoor space': 'outdoor space',
    'common outdoor space': 'outdoor space',
    'patio': 'outdoor space',
    'publicoutdoor': 'outdoor space',
    'garden': 'outdoor space',
    'fitness center': 'fitness center',
    'gym': 'fitness center',
    'gym/fitness': 'fitness center',
    'pool': 'pool',
    'swimming pool': 'pool',
    'parking space': 'parking spot',
    'parking space': 'parking spot',
    'on-site garage': 'parking spot',
    'parking': 'parking spot',
    'garage': 'parking spot',
    'common parking/garage': 'parking spot',
    'furnished': 'furnished'
}

In [18]:
# Create a new dictionary to hold the cleaned and combined counts
cleaned_features_dict = {}

for old_name, count in features_dict.items():
    standard_name = feature_mapping.get(old_name, old_name)

    cleaned_features_dict[standard_name] = cleaned_features_dict.get(standard_name, 0) + count

cleaned_features_sorted = sorted(cleaned_features_dict.items(), key=lambda item: item[1], reverse=True)

for i in range(min(50, len(cleaned_features_sorted))):
    print(cleaned_features_sorted[i])

('pet policy', 45697)
('elevator', 26273)
('hardwood floors', 23558)
('doorman', 21320)
('dishwasher', 20806)
('in building wash/dryer', 20400)
('outdoor space', 19708)
('no fee', 18079)
('fitness center', 13430)
('in unit wash/dryer', 9807)
('pre-war', 9149)
('dining room', 5150)
('high speed internet', 4299)
('pool', 3053)
('new construction', 2608)
('terrace', 2313)
('exclusive', 2167)
('loft', 2101)
('parking spot', 1496)
('wheelchair access', 1358)
('prewar', 1349)
('hardwood', 1058)
('fireplace', 919)
('simplex', 908)
('lowrise', 789)
('reduced fee', 699)
('furnished', 690)
('multi-level', 632)
('high ceilings', 613)
('live in super', 366)
('renovated', 337)
('green building', 247)
('storage', 217)
('high ceiling', 200)
('stainless steel appliances', 177)
('newly renovated', 160)
('light', 146)
('live-in superintendent', 118)
('granite kitchen', 117)
('bike room', 114)
('exposed brick', 113)
('marble bath', 108)
('walk in closet(s)', 105)
('subway', 100)
('residents lounge', 98)


In [27]:
def standardize_features(cell_data):
    if cell_data is None or (isinstance(cell_data, float) and cell_data != cell_data):
        return []

    if isinstance(cell_data, str):
        clean_string = cell_data.strip("[]")
        if not clean_string:
            return []
        feature_list = clean_string.split(',')
    elif isinstance(cell_data, (list, tuple, set)):
        feature_list = list(cell_data)
    else:
        return [cell_data]


    new_features = []
    for f in feature_list:
        clean_name = str(f).strip(" '\"")

        mapped_name = feature_mapping.get(clean_name.lower(), clean_name)
        new_features.append(mapped_name)

    return new_features

df['features'] = df['features'].apply(standardize_features)

print(df['features'].head())

4     [Dining Room, Pre-War, in building wash/dryer,...
6     [doorman, elevator, in building wash/dryer, di...
9     [doorman, elevator, in building wash/dryer, in...
10                                                   []
15    [doorman, elevator, fitness center, in buildin...
Name: features, dtype: object


In [29]:
target_features = set(feature_mapping.values())

for target in target_features:
    df[target] = df['features'].apply(
        lambda feature_list: 1 if isinstance(feature_list, list) and target in feature_list else 0
    )

# 3. (Optional) Drop the original 'features' column since you've extracted everything
# df = df.drop('features', axis=1)

# Check the results!
print(df.head())

    bathrooms  bedrooms                       building_id  \
4         1.0         1  8579a0b0d54db803821a35a4a615e97a   
6         1.0         2  b8e75fc949a6cd8225b455648a951712   
9         1.0         2  cd759a988b8f23924b5a2058d5ab2b49   
10        1.5         3  53a5b119ba8f7b61d4e010512e0dfc85   
15        1.0         0  bfb9405149bfff42a92980b594c28234   

                created                                        description  \
4   2016-06-16 05:55:27  Spacious 1 Bedroom 1 Bathroom in Williamsburg!...   
6   2016-06-01 05:44:33  BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...   
9   2016-06-14 15:19:59  **FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...   
10  2016-06-24 07:54:24  A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...   
15  2016-06-28 03:50:23  Over-sized Studio w abundant closets. Availabl...   

        display_address                                           features  \
4   145 Borinquen Place  [Dining Room, Pre-War, in building wash/dryer,...   
6       

In [33]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
df.head(5)

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level,outdoor space,dishwasher,elevator,pool,furnished,in unit wash/dryer,fitness center,in building wash/dryer,parking spot,pet policy,doorman
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,"Spacious 1 Bedroom 1 Bathroom in Williamsburg!Apartment Features:- Renovated Eat in Kitchen With Dishwasher- Renovated Bathroom- Beautiful Hardwood Floors- Lots of Sunlight- Great Closet Space- Freshly Painted- Heat and Hot Water Included- Live in Super Nearby L, J, M & G Trains !<br /><br />Contact Information:Kenneth BeakExclusive AgentC: 064-692-8838Email: kagglemanager@renthop.com, Text or Email to schedule a private viewing!<br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><p><a website_redacted",145 Borinquen Place,"[Dining Room, Pre-War, in building wash/dryer, dishwasher, Hardwood Floors, pet policy, pet policy]",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,"[https://photos.renthop.com/2/7170325_3bb5ac84a5a10227b17b273e79bd77b4.jpg, https://photos.renthop.com/2/7170325_a29a17a771ee6af213966699b05c8ea2.jpg, https://photos.renthop.com/2/7170325_149a898e8760cac1cad56e30cfe98baa.jpg, https://photos.renthop.com/2/7170325_f74a43d781bcc3c5588e61dd47de81ba.jpg, https://photos.renthop.com/2/7170325_e677d9d249ac99abe01aa5454c6e9f59.jpg, https://photos.renthop.com/2/7170325_960ea0e180bf2f15467b68b455db6172.jpg, https://photos.renthop.com/2/7170325_cbc1b8437155dbf7f5d63b3a0b5a45a3.jpg, https://photos.renthop.com/2/7170325_9a9f2adc2ce922e1d5394727efdf64bb.jpg, https://photos.renthop.com/2/7170325_aae2a39d536103eebb282775fab1c315.jpg, https://photos.renthop.com/2/7170325_cd290d0051b9f08e3482195dcbf6b5a6.jpg, https://photos.renthop.com/2/7170325_a2b599da7880eea1edd10c4b04250dc1.jpg, https://photos.renthop.com/2/7170325_6b83fa82d662bcb09733ac3a8a107113.jpg]",2400,145 Borinquen Place,medium,0,1,0,0,0,0,0,1,0,1,0
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,"BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind yourself and your home in the center of it all. Steps from Grand Central Station, at the epicenter of Manhattan, The Centra combines convenience and luxury to create a perfectly balanced living experience. Offering newly renovated over sized apartment layouts.<br /><br />Full Time DoormanElevatorNewly Renovated HallwaysLaundry in BuildingOn-Site Parking Garage<br /><br />I operate with the utmost care and integrity. The client is my #1 priority. Contact me for a viewing of the great apartment, I'm more than confident we'll find a place for you to call home.Call/Text Keon: Email: If you require a move within 30 days write ""URGENT"" in the subject email or text message to be taken with high priority.<br /><br />One Month Free - net effective rent listed<p><a website_redacted",East 44th,"[doorman, elevator, in building wash/dryer, dishwasher, Hardwood Floors, No Fee]",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,"[https://photos.renthop.com/2/7092344_7663c19af02c46104bc4c569f7162ae0.jpg, https://photos.renthop.com/2/7092344_8287349abe511d195a7b6129bf24af0e.jpg, https://photos.renthop.com/2/7092344_e9e6a2b7aa95aa7564fe3318cadcf4e7.jpg, https://photos.renthop.com/2/7092344_d51ee4b92fd9246633f93afe6e86d8f0.jpg, https://photos.renthop.com/2/7092344_f0573fa184ca130b1b6000f2fa90511c.jpg, https://photos.renthop.com/2/7092344_b2a62f769a59a317b0a243000db46fd0.jpg]",3800,230 East 44th,low,0,1,1,0,0,0,0,1,0,0,1
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,"**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**Looking for the perfect apartment in Midtown East - Sutton Place? Come check out the beautiful apartment in this prime location! **Mid 50's and 1st Ave** Elevator Building with 24-hour doorman, laundry room, and bike room!LARGE living space with King Bedroom!Beautiful large kitchen with stainless stee

In [35]:
df.to_csv("2017_housing_clean.csv")